In [1]:
import json
import numpy as np
import pandas as pd

from uniguide.config import get_path, load_config
from uniguide.evaluation.protocol import evaluate_model
from uniguide.models.popularity import PopularityRecommender

cfg = load_config()
p = get_path("processed")
train, val, test = (pd.read_parquet(p / f"{n}.parquet") for n in ["train", "val", "test"])
courses = pd.read_parquet(p / "courses.parquet")
cold_courses = set(json.loads((p / "cold_courses.json").read_text()))

# Main comparison: the candidate catalog excludes cold-start courses (they never appear in test)
courses_warm = courses[~courses["course_id"].isin(cold_courses)].reset_index(drop=True)
K = tuple(cfg["evaluation"]["k_values"])
names = courses.set_index("course_id")[["name", "institution"]]


def mean_metric(res, metric, k):
    """Mean of one metric at one K over all evaluated users."""
    return res[(res["metric"] == metric) & (res["k"] == k)]["value"].mean()

In [2]:
rows = []
for q in [0.0, 0.25, 0.5, 0.75, 0.9, 0.99]:
    model = PopularityRecommender(m_quantile=q).fit(train, courses_warm)
    res = evaluate_model(model, val, k_values=K, experiment=f"m_quantile={q}")
    rows.append({"m_quantile": q, "m": round(model.m, 1),
                 "ndcg@10": mean_metric(res, "ndcg", 10),
                 "hit_rate@10": mean_metric(res, "hit_rate", 10)})
tuning = pd.DataFrame(rows)
tuning

,m_quantile,m,ndcg@10,hit_rate@10
0,0.00,4.0,0.006304,0.014936
1,0.25,58.8,0.044640,0.100566
2,0.50,108.0,0.056684,0.106294
3,0.75,229.5,0.076477,0.167633
4,0.90,436.2,0.091987,0.196998
5,0.99,1750.1,0.099878,0.214690


In [3]:
best_q = float(tuning.loc[tuning["ndcg@10"].idxmax(), "m_quantile"])
tuning.to_csv(get_path("results") / "tables" / "tuning_popularity.csv", index=False)
best_q

0.99

In [4]:
train_val = pd.concat([train, val])
pop = PopularityRecommender(m_quantile=best_q).fit(train_val, courses_warm)
res_pop = evaluate_model(pop, test, k_values=K, experiment="main", seed=cfg["seed"])
res_pop.to_csv(get_path("results") / "tables" / "eval_popularity.csv", index=False)
print(res_pop.attrs)   # evaluated users / excluded users
res_pop.groupby(["k", "metric"])["value"].mean().unstack()

{'evaluated_users': 14039, 'excluded_users_without_relevant_items': 717}


metric,hit_rate,ndcg,precision,recall
k,,,,
5,0.182135,0.081818,0.041684,0.117964
10,0.279008,0.109250,0.032972,0.196642


In [5]:
pop.stats.sort_values("wr", ascending=False).head(10).join(names)

,v,R,wr,name,institution
course_id,,,,,
python-data,2585,4.845261,4.754081,Python Data Structures,University of Michigan
excel-essentials,1200,4.880833,4.728227,Excel Skills for Business: Essentials,Macquarie University
learning-how-to-learn,2479,4.791448,4.722376,Learning How to Learn: Powerful mental tools t...,University of California San Diego
ai-for-everyone,1286,4.854588,4.722013,AI For Everyone,DeepLearning.AI
neural-networks-deep-learning,2968,4.767183,4.714730,Neural Networks and Deep Learning,DeepLearning.AI
deep-neural-network,1915,4.787990,4.710833,Improving Deep Neural Networks: Hyperparameter...,DeepLearning.AI
technical-support-fundamentals,1858,4.789559,4.710452,Technical Support Fundamentals,Google
machine-learning,2788,4.748207,4.701799,Machine Learning,Stanford University
excel-intermediate-1,509,4.891945,4.688976,Excel Skills for Business: Intermediate I,Macquarie University


In [6]:
user = test["user_id"].iloc[0]
pop.recommend(user, k=10).join(names, on="course_id")

,course_id,score,name,institution
0,python-data,4.754081,Python Data Structures,University of Michigan
1,excel-essentials,4.728227,Excel Skills for Business: Essentials,Macquarie University
2,learning-how-to-learn,4.722376,Learning How to Learn: Powerful mental tools t...,University of California San Diego
3,ai-for-everyone,4.722013,AI For Everyone,DeepLearning.AI
4,technical-support-fundamentals,4.710452,Technical Support Fundamentals,Google
5,machine-learning,4.701799,Machine Learning,Stanford University
6,excel-intermediate-1,4.688976,Excel Skills for Business: Intermediate I,Macquarie University
7,matlab,4.685684,Introduction to Programming with MATLAB,Vanderbilt University
8,mindshift,4.685301,Mindshift: Break Through Obstacles to Learning...,McMaster University
9,python-network-data,4.683107,Using Python to Access Web Data,University of Michigan
